# SU(2) gate diffusion experiments

Scratch notebook for running package experiments against Colab GPU or a local VS Code/Jupyter kernel. Keep project logic in `su2diffusion`; use this notebook to choose configs, run reports, and compare outputs.

In [ ]:
# Run this in Colab or any fresh remote runtime before importing su2diffusion.
# Change BRANCH when testing a PR branch; use "main" after merge.
BRANCH = "codex/hamiltonian-target-demo"

!pip install --no-cache-dir --force-reinstall --no-deps git+https://github.com/joe-singh/su2diffusion.git@{BRANCH}

In [ ]:
from dataclasses import replace

import torch

from su2diffusion import (
    center_names_for_config,
    diagnose_samples,
    get_experiment_config,
    generate_solution_stack_dataset,
    plot_solution_stack_circuit_comparison,
    plot_target_conditioned_circuit_comparison,
    plot_skeleton_local_refinement_comparison,
    plot_target_conditioned_learning_curve,
    plot_target_conditioned_overfit,
    print_solution_stack_circuit_comparison_summary,
    print_target_conditioned_circuit_comparison_summary,
    print_skeleton_local_refinement_summary,
    print_target_conditioned_learning_curve,
    print_target_conditioned_overfit_summary,
    print_solution_stack_dataset_summary,
    run_solution_stack_circuit_experiment,
    run_target_conditioned_circuit_proposal_benchmark,
    run_target_conditioned_learning_curve,
    run_target_conditioned_overfit_diagnostic,
    run_target_conditioned_solution_stack_circuit_experiment,
    run_target_conditioned_synthetic_circuit_experiment,
    run_skeleton_local_refinement_benchmark,
    get_circuit_experiment_config,
    plot_joint_circuit_comparison,
    print_joint_circuit_comparison_summary,
    run_circuit_experiment,
    run_joint_circuit_proposal_benchmark,
    plot_experiment_report,
    plot_hidden_shallow_circuit_best_fidelities,
    plot_hidden_two_entangler_best_fidelities,
    plot_near_clifford_two_entangler_best_fidelities,
    plot_hamiltonian_two_entangler_benchmark,
    plot_hamiltonian_suite,
    print_near_clifford_two_entangler_benchmark,
    print_near_clifford_two_entangler_summary,
    print_hamiltonian_target,
    print_hamiltonian_suite,
    print_hamiltonian_suite_summary,
    print_hamiltonian_two_entangler_benchmark,
    print_hamiltonian_two_entangler_summary,
    run_near_clifford_two_entangler_benchmark,
    make_hamiltonian_target,
    make_random_pauli_hamiltonian_targets,
    run_hamiltonian_suite_benchmark,
    run_hamiltonian_two_entangler_benchmark,
    plot_refinement_ablation,
    print_refinement_ablation_results,
    print_refinement_ablation_summary,
    run_refinement_ablation_benchmark,
    plot_refinement_improvements,
    print_refinement_results,
    print_refinement_summary,
    refine_hidden_two_entangler_benchmark,
    plot_synthesis_fidelity_histograms,
    print_center_mass_table,
    print_conditional_label_table,
    print_diagnostics_table,
    print_per_center_table,
    print_synthesis_candidates,
    print_hidden_shallow_circuit_benchmark,
    print_hidden_shallow_circuit_summary,
    print_hidden_two_entangler_circuit_benchmark,
    print_hidden_two_entangler_circuit_summary,
    print_synthesis_summary,
    resample_experiment,
    run_hidden_shallow_circuit_benchmark,
    run_hidden_two_entangler_circuit_benchmark,
    run_experiment,
    slot_labels_for_named_target,
    synthesize_bell_state,
    synthesize_bell_state_report,
    synthesize_named_gate,
    synthesize_named_gate_label_grid,
    synthesize_named_gate_label_grid_report,
    synthesize_named_gate_report,
    synthesize_named_gate_unconstrained,
    synthesize_named_gate_unconstrained_report,
)
from su2diffusion.data import centers_for_config

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

## Train single-qubit Clifford diffusion

Run the configured experiment and inspect single-gate diagnostics before using the generated gates for synthesis.

In [ ]:
CONFIG_NAME = "baseline-clifford-cond"
EVAL_COUNT = 1000

config = get_experiment_config(CONFIG_NAME)
config = replace(config, sample_count=EVAL_COUNT, reference_count=EVAL_COUNT)
config

In [ ]:
result = run_experiment(config, device=device)

center_names = center_names_for_config(result.config.data)
print_diagnostics_table(result.diagnostics)
print()
print_center_mass_table(result.diagnostics, center_names=center_names)
if result.conditional_diagnostics is not None:
    print()
    print_conditional_label_table(result.conditional_diagnostics, center_names=center_names)
plot_experiment_report(result)

## Named 2-qubit synthesis sanity check

Use generated single-qubit gates as local layers around fixed entanglers for CZ, CNOT, and Bell-state preparation.

In [ ]:
# 2-qubit synthesis demo: generated local gates inside a local-entangler-local template.
local_gates = result.generated_stochastic
local_labels = [center_names[int(i)] for i in result.stochastic_labels]

print("Guided decomposition sanity check")
cz_report = synthesize_named_gate_report(
    local_gates,
    target="cz",
    entangler="cz",
    n_candidates=2000,
    top_k=5,
    local_labels=local_labels,
    slot_label_names=slot_labels_for_named_target("cz", "cz"),
    name="CZ guided",
)
cnot_report = synthesize_named_gate_report(
    local_gates,
    target="cnot",
    entangler="cz",
    n_candidates=2000,
    top_k=5,
    local_labels=local_labels,
    slot_label_names=slot_labels_for_named_target("cnot", "cz"),
    name="CNOT guided",
)
bell_report = synthesize_bell_state_report(
    local_gates,
    entangler="cnot",
    n_candidates=2000,
    top_k=5,
    local_labels=local_labels,
    slot_label_names=("I", "I", "H", "I"),
    name="Bell guided",
)

guided_reports = [cz_report, cnot_report, bell_report]
for title, report in zip(
    ["CZ target via CZ entangler", "CNOT target via CZ entangler", "Bell-state preparation via CNOT entangler"],
    guided_reports,
):
    print(f"\n{title}")
    print_synthesis_candidates(report.candidates)
print("\nGuided summary")
print_synthesis_summary(guided_reports)

print("\nLabel-grid synthesis search over one generated representative per Clifford label")
label_grid_reports = {
    "CZ label grid": synthesize_named_gate_label_grid_report(
        local_gates,
        local_labels,
        target="cz",
        entangler="cz",
        top_k=5,
        name="CZ label grid",
    ),
    "CNOT label grid": synthesize_named_gate_label_grid_report(
        local_gates,
        local_labels,
        target="cnot",
        entangler="cz",
        top_k=5,
        name="CNOT label grid",
    ),
}
for report in label_grid_reports.values():
    print(f"\n{report.name}")
    print_synthesis_candidates(report.candidates)
print("\nLabel-grid summary")
print_synthesis_summary(label_grid_reports)
plot_synthesis_fidelity_histograms(label_grid_reports, bins=60)

print("\nRandom generated-gate search")
random_reports = {
    "CZ random": synthesize_named_gate_unconstrained_report(
        local_gates,
        target="cz",
        entangler="cz",
        n_candidates=100_000,
        top_k=5,
        local_labels=local_labels,
        seed=1,
        name="CZ random",
    ),
    "CNOT random": synthesize_named_gate_unconstrained_report(
        local_gates,
        target="cnot",
        entangler="cz",
        n_candidates=100_000,
        top_k=5,
        local_labels=local_labels,
        seed=2,
        name="CNOT random",
    ),
}
for report in random_reports.values():
    print(f"\n{report.name}")
    print_synthesis_candidates(report.candidates)
print("\nRandom search summary")
print_synthesis_summary(random_reports)
plot_synthesis_fidelity_histograms(random_reports, bins=60)

## Hidden shallow-circuit benchmark

Generate shallow 2-qubit targets from exact Clifford local layers, hide the labels, and recover high-fidelity circuits with generated local gates.

In [ ]:
# Hidden shallow-circuit benchmark: generate targets from exact Clifford local layers,
# hide the local labels, then ask generated gates to recover high-fidelity circuits.
exact_gates = centers_for_config(result.config.data, device=device)
exact_labels = center_names_for_config(result.config.data)

hidden_benchmarks = run_hidden_shallow_circuit_benchmark(
    exact_gates=exact_gates,
    exact_labels=exact_labels,
    generated_gates=local_gates,
    generated_labels=local_labels,
    n_targets=6,
    entangler="cz",
    n_random_candidates=100_000,
    top_k=5,
    seed=42,
)

print_hidden_shallow_circuit_benchmark(hidden_benchmarks)

hidden_label_grid_reports = {
    item.target.name: item.generated_label_grid_report for item in hidden_benchmarks
}
hidden_random_reports = {
    item.target.name: item.random_report for item in hidden_benchmarks
}

print("\nGenerated label-grid summary")
print_synthesis_summary(hidden_label_grid_reports)
plot_synthesis_fidelity_histograms(hidden_label_grid_reports, bins=60)

print("\nGenerated random-search summary")
print_synthesis_summary(hidden_random_reports)
plot_synthesis_fidelity_histograms(hidden_random_reports, bins=60)

In [ ]:
# Larger hidden shallow-circuit benchmark: summarize success rates over many hidden targets.
# This keeps only top candidates and best-fidelity aggregates, not every searched fidelity.
BENCHMARK_TARGETS = 50
BENCHMARK_RANDOM_CANDIDATES = 100_000

hidden_benchmark_suite = run_hidden_shallow_circuit_benchmark(
    exact_gates=exact_gates,
    exact_labels=exact_labels,
    generated_gates=local_gates,
    generated_labels=local_labels,
    n_targets=BENCHMARK_TARGETS,
    entangler="cz",
    n_random_candidates=BENCHMARK_RANDOM_CANDIDATES,
    top_k=5,
    seed=123,
    keep_fidelities=False,
)

print_hidden_shallow_circuit_summary(hidden_benchmark_suite)
plot_hidden_shallow_circuit_best_fidelities(hidden_benchmark_suite)

## Hidden two-entangler benchmark

Increase template depth to local - CZ - local - CZ - local. This uses random search over six generated local-gate slots, since exhaustive Clifford grids are too large at this depth.

In [ ]:
# Two-entangler hidden benchmark: a broader circuit family with six local slots.
TWO_ENTANGLER_TARGETS = 12
TWO_ENTANGLER_RANDOM_CANDIDATES = 200_000

two_entangler_benchmarks = run_hidden_two_entangler_circuit_benchmark(
    exact_gates=exact_gates,
    exact_labels=exact_labels,
    generated_gates=local_gates,
    generated_labels=local_labels,
    n_targets=TWO_ENTANGLER_TARGETS,
    entangler="cz",
    n_random_candidates=TWO_ENTANGLER_RANDOM_CANDIDATES,
    top_k=5,
    seed=321,
    keep_fidelities=False,
)

print_hidden_two_entangler_circuit_benchmark(two_entangler_benchmarks[:6])
print("\nTwo-entangler summary")
print_hidden_two_entangler_circuit_summary(two_entangler_benchmarks)
plot_hidden_two_entangler_best_fidelities(two_entangler_benchmarks)

In [ ]:
# Eta sweep: reuse the trained model to tune within-gate spread without retraining.
ETA_SWEEP = [0.3, 0.5, 0.7, 1.0]
eta_results = resample_experiment(result, ETA_SWEEP, device=device)

print_diagnostics_table({name: item.diagnostics for name, item in eta_results.items()})
print()
print_center_mass_table({name: item.diagnostics for name, item in eta_results.items()}, center_names=center_names)
conditional_eta = {
    name: item.conditional_diagnostics
    for name, item in eta_results.items()
    if item.conditional_diagnostics is not None
}
if conditional_eta:
    print()
    print_conditional_label_table(conditional_eta, center_names=center_names)

## Local refinement of generated two-entangler circuits

This keeps the discovered depth-2 template fixed, then optimizes the six generated local SU(2) gates directly against each hidden target.


In [ ]:
# Refine the best generated-random candidate for each hidden depth-2 target.
refinement_results = refine_hidden_two_entangler_benchmark(
    two_entangler_benchmarks,
    generated_gates=local_gates,
    num_steps=200,
    lr=0.05,
)

print_refinement_results(refinement_results[:6])
print()
print_refinement_summary(refinement_results)
plot_refinement_improvements(refinement_results)


## Refinement ablation

Compare diffusion-seeded refinement against random Haar-seeded refinement with the same optimizer and hidden targets.


In [ ]:
# Honesty check: does the diffusion/search seed beat random SU(2) starting points?
ablation_results = run_refinement_ablation_benchmark(
    two_entangler_benchmarks,
    generated_gates=local_gates,
    generated_results=refinement_results,
    n_random_starts=4,
    num_steps=200,
    lr=0.05,
    seed=0,
)

print_refinement_ablation_results(ablation_results[:6])
print()
print_refinement_ablation_summary(ablation_results)
plot_refinement_ablation(ablation_results)


## Near-Clifford hidden circuits

These targets perturb each hidden Clifford local gate by a small continuous SU(2) update before composing the depth-2 circuit. This makes exact Clifford lookup an intentionally imperfect baseline.


In [ ]:
# Continuous near-Clifford benchmark: exact Clifford lookup vs analytic noisy Cliffords vs diffusion samples vs Haar samples.
near_clifford_benchmarks = run_near_clifford_two_entangler_benchmark(
    clifford_gates=exact_gates,
    clifford_labels=center_names,
    generated_gates=local_gates,
    generated_labels=local_labels,
    n_targets=12,
    perturb_scale=0.12,
    n_random_candidates=200_000,
    n_analytic_gates=1024,
    n_haar_gates=1024,
    top_k=5,
    seed=29,
    keep_fidelities=False,
)

print_near_clifford_two_entangler_benchmark(near_clifford_benchmarks[:6])
print()
print_near_clifford_two_entangler_summary(near_clifford_benchmarks)
plot_near_clifford_two_entangler_best_fidelities(near_clifford_benchmarks)


## Hamiltonian target demo

Build a two-qubit Hamiltonian from Pauli-string terms, exponentiate it to `U(t)`, then synthesize against that unitary with the existing two-entangler proposal baselines.


In [ ]:
hamiltonian_target = make_hamiltonian_target(
    [
        ("XI", 0.35),   # X0 = X tensor I
        ("IZ", -0.22),  # Z1 = I tensor Z
        ("XX", 0.18),   # X0 X1 = X tensor X
        ("ZZ", 0.12),   # Z0 Z1 = Z tensor Z
    ],
    time=0.8,
    name="pauli-demo",
    device=device,
)

hamiltonian_benchmark = run_hamiltonian_two_entangler_benchmark(
    hamiltonian_target,
    clifford_gates=exact_gates,
    clifford_labels=center_names,
    generated_gates=local_gates,
    generated_labels=local_labels,
    perturb_scale=0.12,
    n_random_candidates=200_000,
    n_analytic_gates=1024,
    n_haar_gates=1024,
    top_k=5,
    seed=404,
    keep_fidelities=False,
)

print_hamiltonian_target(hamiltonian_target)
print()
print_hamiltonian_two_entangler_benchmark(hamiltonian_benchmark)
print()
print_hamiltonian_two_entangler_summary(hamiltonian_benchmark)
plot_hamiltonian_two_entangler_benchmark(hamiltonian_benchmark)


In [ ]:
# Small Hamiltonian suite: same Pauli-term family, random coefficients.
hamiltonian_targets = make_random_pauli_hamiltonian_targets(
    n_targets=12,
    terms=("XI", "IZ", "XX", "ZZ"),
    coefficient_scale=0.35,
    time=0.8,
    seed=505,
    device=device,
)

hamiltonian_suite = run_hamiltonian_suite_benchmark(
    hamiltonian_targets,
    clifford_gates=exact_gates,
    clifford_labels=center_names,
    generated_gates=local_gates,
    generated_labels=local_labels,
    perturb_scale=0.12,
    n_random_candidates=100_000,
    n_analytic_gates=1024,
    n_haar_gates=1024,
    top_k=5,
    seed=606,
    keep_fidelities=False,
)

print_hamiltonian_suite(hamiltonian_suite)
print()
print_hamiltonian_suite_summary(hamiltonian_suite)
plot_hamiltonian_suite(hamiltonian_suite)


## Joint circuit diffusion

Train a model whose samples are full six-gate `SU(2)^6` circuit stacks, then compare those joint proposals against the single-gate proposal baselines on the same hidden near-Clifford targets.


In [ ]:
CIRCUIT_CONFIG_NAME = "baseline-circuit-near-clifford"
CIRCUIT_SAMPLE_COUNT = 1000

circuit_config = get_circuit_experiment_config(CIRCUIT_CONFIG_NAME)
circuit_config = replace(circuit_config, sample_count=CIRCUIT_SAMPLE_COUNT)
circuit_result = run_circuit_experiment(circuit_config, device=device)

joint_reports = run_joint_circuit_proposal_benchmark(
    near_clifford_benchmarks,
    circuit_stacks=circuit_result.generated_stochastic,
    top_k=5,
    keep_fidelities=False,
)

print_joint_circuit_comparison_summary(near_clifford_benchmarks, joint_reports)
plot_joint_circuit_comparison(near_clifford_benchmarks, joint_reports)


## Optional: solution-stack circuit diffusion

This was useful as a negative-control branch, but it is slow and not needed for the default target-conditioned run. Leave `RUN_OPTIONAL_SOLUTION_STACK = False` unless you specifically want to reproduce that comparison.


In [ ]:
RUN_OPTIONAL_SOLUTION_STACK = False

if RUN_OPTIONAL_SOLUTION_STACK:
    solution_dataset = generate_solution_stack_dataset(
        clifford_gates=exact_gates,
        clifford_labels=center_names,
        n_targets=256,
        perturb_scale=0.12,
        candidate_count=512,
        refinement_steps=100,
        fidelity_threshold=0.995,
        seed=101,
    )
    print_solution_stack_dataset_summary(solution_dataset)

    solution_circuit_result = run_solution_stack_circuit_experiment(
        solution_dataset.stacks,
        circuit_config,
        device=device,
    )

    solution_joint_reports = run_joint_circuit_proposal_benchmark(
        near_clifford_benchmarks,
        circuit_stacks=solution_circuit_result.generated_stochastic,
        top_k=5,
        keep_fidelities=False,
    )

    print_solution_stack_circuit_comparison_summary(
        near_clifford_benchmarks,
        joint_reports,
        solution_joint_reports,
    )
    plot_solution_stack_circuit_comparison(near_clifford_benchmarks, joint_reports, solution_joint_reports)


## Target-conditioned circuit diffusion

Train on fresh synthetic `(target unitary, six-gate stack)` pairs generated on the fly. At sampling time, each hidden benchmark target gets its own batch of generated six-gate stacks.


In [ ]:
eval_target_unitaries = torch.stack([item.target.unitary for item in near_clifford_benchmarks])

target_conditioned_result = run_target_conditioned_synthetic_circuit_experiment(
    eval_target_unitaries,
    circuit_config,
    device=device,
)

target_conditioned_reports = run_target_conditioned_circuit_proposal_benchmark(
    near_clifford_benchmarks,
    circuit_stacks_by_target=target_conditioned_result.generated_stochastic_by_target,
    top_k=5,
    keep_fidelities=False,
)

print_target_conditioned_circuit_comparison_summary(
    near_clifford_benchmarks,
    joint_reports,
    target_conditioned_reports,
)
plot_target_conditioned_circuit_comparison(
    near_clifford_benchmarks,
    joint_reports,
    target_conditioned_reports,
)


## Skeleton-local circuit diffusion

Condition the circuit denoiser on both the target unitary and the six Clifford skeleton labels, then compare two starts from the same trained model: a global Haar start and a local start around the requested Clifford skeleton.


In [ ]:
skeleton_local_result = run_skeleton_local_refinement_benchmark(
    near_clifford_benchmarks,
    circuit_config,
    device=device,
    init_noise_scale=0.05,
)

print_skeleton_local_refinement_summary(
    near_clifford_benchmarks,
    joint_reports,
    target_conditioned_reports,
    skeleton_local_result.global_reports,
    skeleton_local_result.local_reports,
)
plot_skeleton_local_refinement_comparison(
    near_clifford_benchmarks,
    joint_reports,
    target_conditioned_reports,
    skeleton_local_result.global_reports,
    skeleton_local_result.local_reports,
)


## Tiny overfit diagnostic

This is the main debugging cell for the current target-conditioned failure. It trains on only a few fixed `(target unitary, stack)` pairs and tests whether the reverse sampler can reproduce those same targets.


In [ ]:
overfit_config = replace(
    circuit_config,
    sample_count=512,
    train=replace(circuit_config.train, num_steps=1000),
)

overfit_result = run_target_conditioned_overfit_diagnostic(
    overfit_config,
    n_targets=4,
    perturb_scale=0.12,
    device=device,
    seed=202,
)

print_target_conditioned_overfit_summary(overfit_result)
plot_target_conditioned_overfit(overfit_result)


## Target-conditioned learning curve

Train separate target-conditioned models on increasing numbers of fixed target/stack pairs, then evaluate both memorization on those training targets and generalization to held-out targets.


In [ ]:
learning_curve_config = replace(
    circuit_config,
    sample_count=256,
    train=replace(circuit_config.train, num_steps=600),
)

learning_curve = run_target_conditioned_learning_curve(
    learning_curve_config,
    target_counts=(1, 4, 16, 64),
    n_heldout_targets=12,
    perturb_scale=0.12,
    device=device,
    seed=303,
)

print_target_conditioned_learning_curve(learning_curve)
plot_target_conditioned_learning_curve(learning_curve)


## Optional diagnostics

Eta sweeps and deeper per-center diagnostics are useful after the main benchmark output looks interesting.

In [ ]:
# Optional: slower, center-by-center gate diagnostics. Run only when the standard report looks interesting.
centers = centers_for_config(result.config.data, device=device)
deep_diagnostics = {
    name: diagnose_samples(
        samples,
        result.clean_reference,
        result.haar_reference,
        centers=centers,
        include_per_center=True,
    )
    for name, samples in {
        "deterministic": result.generated_deterministic,
        "stochastic": result.generated_stochastic,
    }.items()
}

print_diagnostics_table(deep_diagnostics)
print_per_center_table(deep_diagnostics, center_names=center_names)

In [ ]:
# Quick config comparison loop. Keep this to smoke/medium configs unless you want a long run.
for name in ["smoke", "smoke-gates", "smoke-gates-cond", "smoke-clifford-cond"]:
    cfg = get_experiment_config(name)
    cfg = replace(cfg, sample_count=512, reference_count=512)
    print("\n", cfg.name, cfg.data.kind, cfg.schedule.kind)
    quick = run_experiment(cfg, device=device)
    names = center_names_for_config(quick.config.data)
    print_diagnostics_table(quick.diagnostics)
    print_center_mass_table(quick.diagnostics, center_names=names)
    plot_experiment_report(quick)